**Imports and data set up**

In [ ]:
import numpy as np
import csv
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

# Raw data inputs
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

TASK 1: Cleaning Machakos Data with try/except and Imputation

In [ ]:
def to_int_or_none(val):
    """Strips whitespace and converts to int; raises ValueError if invalid."""
    cleaned = str(val).strip()
    if cleaned == "":
        raise ValueError("Empty string")
    return int(cleaned)

# First pass: Parse valid values to calculate the mean for imputation
valid_machakos = []
parsed_machakos_raw = []

for idx, item in enumerate(machakos_raw):
    try:
        val = to_int_or_none(item)
        valid_machakos.append(val)
        parsed_machakos_raw.append(val)
    except ValueError:
        parsed_machakos_raw.append(None)

# Calculate rounded average of valid values for imputation
imputation_value = int(round(sum(valid_machakos) / len(valid_machakos)))

# Second pass: Replace None with imputation value and track imputed months
machakos_cleaned = []
imputed_months = []

for idx, val in enumerate(parsed_machakos_raw):
    if val is None:
        machakos_cleaned.append(imputation_value)
        imputed_months.append(months[idx])
    else:
        machakos_cleaned.append(val)

print(f"Imputed months for Machakos: {imputed_months}")
print(f"Imputed replacement value: {imputation_value} mm")
print(f"Cleaned Machakos row: {machakos_cleaned}")

Imputed months for Machakos: ['April', 'September']
Imputed replacement value: 68 mm
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


**Task 2:combine into single 5 x12 Numpy array**

In [ ]:

all_data = clean_rows + [machakos_cleaned]
rainfall_matrix = np.array(all_data, dtype=float)

print("Matrix Shape:", rainfall_matrix.shape)
print("Matrix Dtype:", rainfall_matrix.dtype)

Matrix Shape: (5, 12)
Matrix Dtype: float64


**task 3:total rainfall per county and the average rainfall per month across all counties**


In [4]:
annual_totals = rainfall_matrix.sum(axis=1)
monthly_averages = rainfall_matrix.mean(axis=0)

print("Annual Totals per County:", annual_totals)
print("Monthly Averages across All Counties:", monthly_averages)

Annual Totals per County: [1045.  916. 1371. 1075.  811.]
Monthly Averages across All Counties: [ 47.   47.6  91.  164.4 171.2  67.6  56.6  58.2  64.2  67.8 127.4  80.6]


**Task 4: Find the Wettest Month per County Using np.argmax**

In [5]:
wettest_month_indices = np.argmax(rainfall_matrix, axis=1)
wettest_months = [months[i] for i in wettest_month_indices]
for county, total, wettest in zip(counties, annual_totals, wettest_months):
    print(f"{county}: Annual Total = {total:.1f} mm | Wettest Month = {wettest}")

Nairobi: Annual Total = 1045.0 mm | Wettest Month = April
Nakuru: Annual Total = 916.0 mm | Wettest Month = April
Kisumu: Annual Total = 1371.0 mm | Wettest Month = April
Mombasa: Annual Total = 1075.0 mm | Wettest Month = May
Machakos: Annual Total = 811.0 mm | Wettest Month = November


**Task 5 — Count Dry Months (< 50 mm)**

In [6]:
dry_months_counts = (rainfall_matrix < 50).sum(axis=1)

for county, count in zip(counties, dry_months_counts):
    print(f"{county}: {count} dry months")

Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


**Task 6 — Recommended Planting Window (Shifted-Array Technique)**

In [7]:
consecutive_two_months = rainfall_matrix[:, :-1] + rainfall_matrix[:, 1:]
best_window_start_indices = np.argmax(consecutive_two_months, axis=1)
planting_windows = []
for idx in best_window_start_indices:
    m1 = months[idx]
    m2 = months[idx + 1]
    planting_windows.append(f"{m1}-{m2}")

for county, window in zip(counties, planting_windows):
    print(f"{county} Recommended Planting Window: {window}")

Nairobi Recommended Planting Window: April-May
Nakuru Recommended Planting Window: April-May
Kisumu Recommended Planting Window: April-May
Mombasa Recommended Planting Window: April-May
Machakos Recommended Planting Window: November-December


**Task 7 — Write rainfall_summary.csv**

In [9]:
csv_filename = "rainfall_summary.csv"
fields = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    for i in range(len(counties)):
        writer.writerow({
            "county": counties[i],
            "annual_total_mm": f"{annual_totals[i]:.1f}",
            "wettest_month": wettest_months[i],
            "dry_months": int(dry_months_counts[i]),
            "planting_window": planting_windows[i]
        })

print(f"Successfully generated {csv_filename} with {len(counties)} rows.")

Successfully generated rainfall_summary.csv with 5 rows.


**TASK 8:ADVISORY TO THE NGO**

PLANTING ADVISORY


* For regions like Machakos, Nairobi, and Nakuru, the optimal sowing period aligns with the onset of the long rains during March and April, capitalizing on initial moisture accumulation.


*   In contrast, lakeside and coastal regions like Kisumu and Mombasa experience prolonged heavy precipitation windows lasting through April and May, requiring flood-resilient crop varieties and robust field drainage planning.

* Field extension officers should coordinate SMS broadcast schedules two weeks prior to these identified windows so farmers can complete land preparation and tillage on time.

DATA QUALITY NOTES


* The primary operational challenge encountered during this analysis stemmed from manual data entry anomalies in the Machakos weather station log, which included text formatting issues, empty strings, and `"N/A"` placeholders. These missing values were successfully resolved using a robust `try-except` parsing filter and imputed with the rounded arithmetic mean of valid station records. However, data-quality caveats remain: imputation assumes uniform meteorological conditions across unrecorded months, which can mask localized microclimate anomalies or sudden weather shocks

*  NGOs must treat imputed estimates with caution and combine statistical models with real-time rain gauge spot-checks.





